# Relationship Condition Development and Production Audit

This notebook is the human-readable control surface for the relationship-condition workstream of the US-China Closeness Index project. Python scripts remain the computational source of truth. The notebook runs them in order, records methodological decisions, inspects outputs and stops when an integrity gate fails.

The analytical unit is **country x anchor x year**, where the anchor is China or the United States. Relationship condition is separate from structural ties.

## Four-state hierarchy

| Condition | Severity | Interpretation |
|---|---:|---|
| `normal` | 0 | No qualifying sanctions or explicit diplomatic disruption |
| `strained` | 1 | Active adverse evidence without a severe trigger |
| `severely_strained` | 2 | Complete or compounded sanctions evidence, or sanctions combined with unilateral diplomatic disruption |
| `ruptured` | 3 | Explicit adverse diplomatic-representation coding in both directions |

`ruptured` denotes bilateral operational disruption of diplomatic representation, not necessarily formal legal severance. Classification uses a controlling trigger, while all contributing triggers remain available for explanation.

## Development chronology

1. Inventoried sanctions and diplomatic-representation sources.
2. Diagnosed GSDB identifiers and years.
3. Distinguished explicit diplomatic adversity from ordinary rank changes.
4. Tested candidate classifications.
5. Corrected non-state sanctions targets and China-US anchor records.
6. Preserved sanctions direction and trade scope upstream.
7. Built a production evidence stage consuming the corrected GSDB panel.
8. Resolved DRD names and retained Serbia and Montenegro as historical `SCG` for 2000-2006.
9. Built the symmetric year-specific production grid and final four-state classifier.

In [1]:
# locate repository and define paths
from pathlib import Path
import subprocess
import sys
import pandas as pd


def locate_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "data").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")


REPOSITORY_ROOT = locate_repository_root()
SRC_DATA = REPOSITORY_ROOT / "src" / "data"
PROCESSED_DATA = REPOSITORY_ROOT / "data" / "processed"
GSDB_OUTPUTS = REPOSITORY_ROOT / "outputs" / "tables" / "gsdb"
CONDITION_OUTPUTS = REPOSITORY_ROOT / "outputs" / "tables" / "relations_condition"
print("Repository root: repository root")

Repository root: repository root


In [2]:
# inventory scripts and key outputs
required_scripts = {
    "GSDB builder": SRC_DATA / "build_gsdb_active_case_panel.py",
    "Evidence builder": SRC_DATA / "build_relationship_condition_evidence.py",
    "Condition classifier": SRC_DATA / "build_relationship_condition_panel.py",
}
key_outputs = {
    "GSDB panel": GSDB_OUTPUTS / "gsdb_country_anchor_year_corrected.csv",
    "Non-country audit": GSDB_OUTPUTS / "gsdb_non_country_entities_excluded.csv",
    "Evidence panel": CONDITION_OUTPUTS / "condition_evidence_panel_harmonized.csv",
    "Production panel": PROCESSED_DATA / "relationship_condition_country_anchor_year.csv",
}
pd.DataFrame([
    {"artefact": k, "path": v.relative_to(REPOSITORY_ROOT).as_posix(), "exists": v.exists()}
    for k,v in {**required_scripts,**key_outputs}.items()
])

,artefact,path,exists
0,GSDB builder,src/data/build_gsdb_active_case_panel.py,True
1,Evidence builder,src/data/build_relationship_condition_evidence.py,True
2,Condition classifier,src/data/build_relationship_condition_panel.py,True
3,GSDB panel,outputs/tables/gsdb/gsdb_country_anchor_year_c...,True
4,Non-country audit,outputs/tables/gsdb/gsdb_non_country_entities_...,True
5,Evidence panel,outputs/tables/relations_condition/condition_e...,True
6,Production panel,data/processed/relationship_condition_country_...,True


In [3]:
# reusable script runner
def run_project_script(script_path: Path) -> str:
    completed = subprocess.run(
        [sys.executable, str(script_path)], cwd=REPOSITORY_ROOT,
        check=False, text=True, capture_output=True,
    )
    if completed.stdout.strip(): print(completed.stdout)
    if completed.stderr.strip():
        print("SCRIPT STDERR:")
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"{script_path.name} failed with exit code {completed.returncode}.")
    return completed.stdout

# Part I. Corrected GSDB panel

The builder audits and excludes recognised non-state targets, stops on unknown country-side entities, preserves valid China-US anchor-to-anchor records, and retains sanctions direction and trade-scope evidence.

In [4]:
# run corrected GSDB stage
gsdb_stdout = run_project_script(required_scripts["GSDB builder"])

GSDB CORRECTED ACTIVE-CASE PANEL
{
  "study_period": [
    2000,
    2025
  ],
  "r5": {
    "dyad_rows": 3943,
    "country_anchor_years": 3764,
    "bridge": {
      "dataset": "R5",
      "bridge_rows": 7583,
      "bridge_unique_case_ids": 629,
      "invalid_against_r5_master_rows": 0,
      "invalid_against_r5_master_unique_ids": 0,
      "rows_in_dataset_subset": 7583,
      "unique_ids_in_dataset_subset": 629,
      "rows_outside_dataset_subset": 0,
      "unique_ids_outside_dataset_subset": 0,
      "duplicate_bridge_rows": 0
    },
    "active_case_count_max": 27,
    "distinct_family_count_max": 6
  },
  "fs": {
    "dyad_rows": 2108,
    "country_anchor_years": 2084,
    "bridge": {
      "dataset": "FS",
      "bridge_rows": 5607,
      "bridge_unique_case_ids": 576,
      "invalid_against_r5_master_rows": 0,
      "invalid_against_r5_master_unique_ids": 0,
      "rows_in_dataset_subset": 4189,
      "unique_ids_in_dataset_subset": 426,
      "rows_outside_dataset_subset":

In [5]:
# validate GSDB stage
gsdb = pd.read_csv(key_outputs["GSDB panel"], low_memory=False)
excluded = pd.read_csv(key_outputs["Non-country audit"], low_memory=False)
required_fields = {"r5_sanctions_directions","r5_trade_exp_partial","r5_trade_exp_complete","r5_trade_imp_partial","r5_trade_imp_complete"}
assert not required_fields.difference(gsdb.columns)
assert gsdb["country"].astype("string").str.fullmatch(r"[A-Z]{3}",na=False).all()
assert not gsdb.duplicated(["country","anchor","year"]).any()
assert {"696","924","925","1023"}.issubset(set(excluded["underlying_case_id"].astype(str)))
assert set(excluded["exclusion_reason"]) == {"non_state_sanctions_target"}
print("PASS: corrected GSDB keys, directions, trade scope and non-state audit")
display(excluded[["underlying_case_id","country_side_entity","exclusion_reason"]].drop_duplicates())

PASS: corrected GSDB keys, directions, trade scope and non-state audit


,underlying_case_id,country_side_entity,exclusion_reason
0,696,Terrorist Organizations (Taliban and Al-Qaeda),non_state_sanctions_target
19,924,Terrorist Organizations (Al-Qaeda),non_state_sanctions_target
20,925,Terrorist Organizations (Taliban),non_state_sanctions_target
32,1023,Terrorist Organizations (ISIL and ANF),non_state_sanctions_target


# Part II. Harmonised evidence

The evidence builder consumes the corrected GSDB panel rather than reopening raw dyads. It resolves DRD names, preserves all contributing triggers and restricts historical `SCG` to 2000-2006.

In [6]:
# run harmonised evidence stage
evidence_stdout = run_project_script(required_scripts["Evidence builder"])

RELATIONSHIP CONDITION EVIDENCE PANEL
{
  "study_period": [
    2000,
    2024
  ],
  "rows": 8918,
  "countries": 199,
  "duplicate_keys": 0,
  "invalid_country_keys": 0,
  "resolved_drd_names": 196,
  "unresolved_drd_names": 0,
  "historical_entities": [
    "Serbia and Montenegro"
  ],
  "condition_counts": {
    "normal": 5331,
    "strained": 3302,
    "severely_strained": 284,
    "ruptured": 1
  },
  "rupture_rows": 1,
  "source_crosswalk_rows": 403
}

Outputs written to: outputs/tables/relations_condition



In [7]:
# validate evidence stage
evidence = pd.read_csv(key_outputs["Evidence panel"], low_memory=False)
unresolved = pd.read_csv(CONDITION_OUTPUTS / "unresolved_drd_country_names.csv")
assert evidence["country"].astype("string").str.fullmatch(r"[A-Z]{3}",na=False).all()
assert not evidence.duplicated(["country","anchor","year"]).any()
assert unresolved.empty
assert "all_condition_triggers" in evidence.columns
scg=evidence[evidence["country"].eq("SCG")]
assert not scg.empty and scg["year"].between(2000,2006).all()
print("PASS: resolved names, unique keys, historical entity and trigger preservation")
display(evidence["condition_four_candidate"].value_counts().rename_axis("candidate_condition").reset_index(name="observations"))

PASS: resolved names, unique keys, historical entity and trigger preservation


,candidate_condition,observations
0,normal,5331
1,strained,3302
2,severely_strained,284
3,ruptured,1


In [8]:
# verify rupture evidence
rupture_evidence=evidence[evidence["drd_bilateral_disruption"]]
assert len(rupture_evidence)==1
r=rupture_evidence.iloc[0]
assert (r["country"],r["anchor"],int(r["year"])) == ("AFG","United States",2021)
assert "bilateral_operational_diplomatic_rupture" in r["all_condition_triggers"]
print("PASS: AFG | United States | 2021 is the sole bilateral rupture evidence row")
display(rupture_evidence[["country","anchor","year","all_condition_triggers"]])

PASS: AFG | United States | 2021 is the sole bilateral rupture evidence row


,country,anchor,year,all_condition_triggers
7607,AFG,United States,2021,sanctions_plus_diplomatic_disruption|bilateral...


# Part III. Final production classifier

For each year, the classifier includes both anchor comparisons for every country observed for either anchor, excluding anchor self-relations and restricting historical entities to their documented period. Canonical names, controlling triggers and all contributing triggers are retained separately.

In [9]:
# run final classifier
production_stdout = run_project_script(required_scripts["Condition classifier"])

RELATIONSHIP CONDITION PRODUCTION PANEL
{
  "rows": 9612,
  "countries_or_historical_entities": 199,
  "duplicate_keys": 0,
  "blank_country_names": 0,
  "severity_mismatches": 0,
  "rupture_without_bilateral_disruption": 0,
  "bilateral_disruption_without_rupture": 0,
  "wrong_rupture_controller": 0,
  "missing_all_condition_triggers": 0,
  "historical_period_violations": 0,
  "condition_counts": {
    "normal": 6025,
    "strained": 3302,
    "severely_strained": 284,
    "ruptured": 1
  }
}

Outputs written to: data/processed/relationship_condition_country_anchor_year.csv
Reports written to: outputs/tables/relations_condition



In [10]:
# validate production panel
production=pd.read_csv(key_outputs["Production panel"],low_memory=False)
coverage=pd.read_csv(CONDITION_OUTPUTS / "relationship_condition_grid_coverage.csv")
required={"country_iso3","country_name","historical_entity","anchor","year","relationship_condition","relationship_condition_severity","controlling_condition_trigger","all_condition_triggers"}
assert not required.difference(production.columns)
assert not production.duplicated(["country_iso3","anchor","year"]).any()
assert production["country_name"].notna().all()
severity={"normal":0,"strained":1,"severely_strained":2,"ruptured":3}
assert (production["relationship_condition"].map(severity)==production["relationship_condition_severity"]).all()
assert production["all_condition_triggers"].astype("string").str.strip().ne("").all()
pivot=coverage.pivot(index="year",columns="anchor",values="countries")
assert pivot["China"].eq(pivot["United States"]).all()
print("PASS: canonical names, unique keys, severity and symmetric annual grids")
display(production["relationship_condition"].value_counts().rename_axis("condition").reset_index(name="observations"))
display(coverage.tail(10))

PASS: canonical names, unique keys, severity and symmetric annual grids


,condition,observations
0,normal,6025
1,strained,3302
2,severely_strained,284
3,ruptured,1


,year,anchor,country_anchor_years,countries,evidence_rows,grid_filled_rows
40,2020,China,196,196,173,23
41,2020,United States,196,196,195,1
42,2021,China,196,196,175,21
43,2021,United States,196,196,195,1
44,2022,China,196,196,176,20
45,2022,United States,196,196,195,1
46,2023,China,196,196,177,19
47,2023,United States,196,196,195,1
48,2024,China,196,196,178,18
49,2024,United States,196,196,195,1


In [11]:
# validate precedence and sole rupture
ruptures=production[production["relationship_condition"].eq("ruptured")]
assert len(ruptures)==1
r=ruptures.iloc[0]
assert (r["country_iso3"],r["country_name"],r["anchor"],int(r["year"])) == ("AFG","Afghanistan","United States",2021)
assert r["drd_bilateral_disruption"]
assert r["controlling_condition_trigger"] == "bilateral_operational_diplomatic_rupture"
assert "bilateral_operational_diplomatic_rupture" in r["all_condition_triggers"]
assert production[production["drd_bilateral_disruption"] & ~production["relationship_condition"].eq("ruptured")].empty
assert production[production["relationship_condition"].eq("ruptured") & ~production["drd_bilateral_disruption"]].empty
print("PASS: rupture logic agrees in both directions")
display(ruptures)

PASS: rupture logic agrees in both directions


,country_iso3,country_name,historical_entity,first_valid_year,last_valid_year,anchor,year,relationship_condition,relationship_condition_severity,controlling_condition_trigger,...,complete_trade_both,reciprocal_sanctions,payment_infrastructure_combination,sanctions_plus_drd_disruption,reciprocal_severe_qualifier,family_combination,trade_scope_profile,combined_evidence_profile,sanctions_directions,evidence_merge_status
8045,AFG,Afghanistan,False,NaN,NaN,United States,2021,ruptured,3,bilateral_operational_diplomatic_rupture,...,False,False,False,True,False,arms+military+financial+travel+other,none,sanctions_plus_bilateral_drd_disruption,anchor_to_country,both


## Production conclusion

The panel now has canonical display names, explicit historical-entity handling, a symmetric annual comparison grid, complete trigger preservation and a controlling trigger consistent with the hierarchy.

Grid-added rows carry `evidence_merge_status = left_only`. They are classified as normal only where the same country-year is observed for the opposite anchor and no adverse evidence is present for the added relationship.

# Completion checklist

- [x] Correct non-state GSDB handling.
- [x] Preserve sanctions directions and trade scope.
- [x] Build production relationship evidence.
- [x] Resolve DRD names and historical `SCG`.
- [x] Preserve all contributing triggers.
- [x] Add canonical country names.
- [x] Construct the year-specific comparison universe.
- [x] Separate controlling and contributing triggers.
- [x] Validate condition, severity and rupture precedence.
- [ ] Update the output manifest and archive superseded diagnostics.

# Decision register

## Decided
- Scripts are the computational source of truth; notebooks orchestrate and explain.
- Non-state sanctions targets are audited and excluded from country panels.
- Downstream evidence consumes corrected upstream outputs.
- `SCG` is retained only for 2000-2006.
- Rupture is bilateral operational diplomatic disruption, not necessarily formal severance.
- The production universe is symmetric by anchor within year, excluding self-relations.
- All contributing triggers are preserved independently from the controlling trigger.

## Next
- Produce the output manifest.
- Classify retained outputs as production, audit, diagnostic or superseded.
- Populate source README files.
- Stage repository changes intentionally.